# Semantic Memory

> **Extract and store generalized facts from conversations, building a persistent knowledge base that transcends any single interaction.**

You know that Paris is the capital of France. But you probably can't remember the exact moment you learned it. That's semantic memory at work: knowledge distilled from experience into stable, context-free facts. The "when" and "how" fade away. The fact itself remains.

AI agents face the same challenge. A user mentions their favorite language is Python in session one, their team size in session two, and their deployment target in session three. Without semantic memory, each session starts from zero. With it, the agent builds a growing profile of distilled facts. Over time it stops asking the same questions. It anticipates needs based on what it already knows.

This notebook shows you how to build a semantic memory system from scratch using the **OpenAI SDK**. You'll extract facts from conversations with an LLM call, store them with vector embeddings (numerical representations of meaning), detect duplicates and contradictions, and retrieve relevant facts for future prompts.

**By the end you'll understand:**
- How to extract clean, declarative facts from messy conversation text.
- How embedding similarity catches duplicates that string matching misses.
- How to detect and resolve contradictions when users change their minds.
- When semantic memory helps and when it quietly fails.

## Key Concepts

- **Semantic memory**: A store of general facts and knowledge, separate from specific episodes. In cognitive science, Endel Tulving coined this term in 1972 to distinguish "knowing that" from "remembering when."
- **Fact extraction**: Pulling declarative statements from conversation text using an LLM prompt. "Oh yeah, I switched to a Mac last month" becomes the fact: "User uses macOS."
- **Embedding**: A list of numbers (a vector) that captures the meaning of a piece of text. Texts with similar meaning have embeddings that point in similar directions. We use embeddings to compare facts by meaning, not by exact wording.
- **Cosine similarity**: A measure of how similar two vectors are. It ranges from -1 (opposite) to 1 (identical direction). We use it to compare embeddings.
- **Deduplication**: Detecting when a new fact means the same thing as one already stored. "I use a Mac" and "My laptop runs macOS" are the same fact. Cosine similarity catches this even though the words differ.
- **Contradiction detection**: Identifying when a new fact conflicts with an existing one. "I use Windows" contradicts a stored "User uses macOS." The system must decide which to keep (usually the newer one).
- **Confidence score**: A number from 0.0 to 1.0 that represents how certain we are about a fact. Explicit statements get high confidence. Inferred facts get lower confidence. Repeated mentions boost confidence.
- **Context injection**: Inserting retrieved facts into the system prompt so the LLM can use them when generating a response.

## Architecture

<p align="center">
 <img src="../../images/diagrams/10_semantic_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
 Conv["Conversation"] --> Extractor["Fact Extractor\n(LLM call)"]
 Extractor --> DedupCheck["Dedup / Conflict\nChecker"]
 DedupCheck -- "new fact" --> KB["Knowledge Base\n(facts + confidence\n+ timestamps)"]
 DedupCheck -- "duplicate" --> Merge["Merge / Update\nConfidence"]
 Merge --> KB
 DedupCheck -- "contradiction" --> Resolve["Resolve Conflict\n(prefer recent)"]
 Resolve --> KB
 NewQuery["New Query"] --> Search["Semantic Search\n(embedding similarity)"]
 KB --> Search
 Search --> Injection["Context Injection"]
 Injection --> LLM["LLM"]
 LLM --> Response["Response"]
```

</details>

The diagram shows the two main flows. **Write path (left to right):** after each conversation, a fact extractor (an LLM call) pulls out declarative facts. Each fact runs through a dedup/conflict checker that compares it against the existing knowledge base. New facts are added. Duplicates merge and boost confidence. Contradictions are resolved by keeping the newer statement.

**Read path (bottom):** when the agent needs to respond, it embeds the current query, searches the knowledge base for relevant facts, and injects them into the prompt. The LLM sees accumulated knowledge from all past sessions.

## Setup

Install dependencies and configure API access. We use the OpenAI SDK for both chat completions and embeddings.

In [ ]:
%pip install -q openai python-dotenv numpy

Import all libraries and load the API key from a `.env` file.

In [ ]:
import os
import json
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone

import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI() # reads OPENAI_API_KEY from environment
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

## Implementation

We'll build a `SemanticMemory` class in four parts:

1. **Fact data structure** and a helper for cosine similarity.
2. **Embedding and extraction**: turning conversations into facts with vector representations.
3. **Deduplication and contradiction detection**: comparing new facts against existing ones.
4. **Retrieval and chat**: fetching relevant facts and injecting them into prompts.

### Part 1: Fact data structure

Think of a filing cabinet where each card holds one fact. Every card has the fact itself, a confidence score, timestamps, and an embedding for search. We also need a quick way to measure how similar two cards are. That's cosine similarity.

In [ ]:
@dataclass
class Fact:
 """A single piece of knowledge extracted from conversation."""
 content: str # e.g., "User prefers dark mode"
 confidence: float = 0.8 # 0.0 to 1.0
 fact_id: str = field(default_factory=lambda: str(uuid.uuid4()))
 created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
 last_confirmed: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
 mention_count: int = 1
 embedding: list[float] = field(default_factory=list)
 archived: bool = False # True if superseded by a newer fact


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
 """Compute cosine similarity between two vectors."""
 a = np.array(vec_a)
 b = np.array(vec_b)
 dot = np.dot(a, b)
 norm = np.linalg.norm(a) * np.linalg.norm(b)
 if norm == 0:
 return 0.0
 return float(dot / norm)

### Part 2: SemanticMemory core, embedding, and fact extraction

The `SemanticMemory` class stores a list of `Fact` objects. It uses the OpenAI embeddings API to convert text into vectors and a chat completion call to extract facts from conversation messages.

The extraction prompt is the heart of the system. It tells the LLM to pull out clean declarative statements and ignore chitchat, questions, and hypotheticals.

In [ ]:
class SemanticMemory:
 """Persistent knowledge base that extracts and stores facts from conversations."""

 EXTRACTION_PROMPT = """You are a fact extractor. Read the conversation below and extract \
declarative facts about the user, their preferences, their work, or their world.

Rules:
- Output one fact per line.
- Each fact must be a short, standalone statement (under 20 words).
- Write in third person: "User ..." or "User's ...".
- Only extract facts that were clearly stated or strongly implied.
- Skip greetings, questions, opinions phrased as questions, and hypotheticals.
- If there are no extractable facts, output exactly: NONE

Conversation:
{conversation}

Extracted facts (one per line):"""

 CONTRADICTION_PROMPT = """Do these two facts contradict each other? \
Answer only YES or NO.

Fact A: {fact_a}
Fact B: {fact_b}

Answer:"""

 def __init__(
 self,
 similarity_threshold: float = 0.85,
 contradiction_threshold: float = 0.50,
 ):
 self.facts: list[Fact] = []
 self.similarity_threshold = similarity_threshold
 self.contradiction_threshold = contradiction_threshold

 # -- Embedding -------------------------------------------------------
 def _embed(self, text: str) -> list[float]:
 """Get an embedding vector for a piece of text."""
 response = client.embeddings.create(
 model=EMBEDDING_MODEL,
 input=text,
 )
 return response.data[0].embedding

 def _embed_batch(self, texts: list[str]) -> list[list[float]]:
 """Embed multiple texts in one API call."""
 response = client.embeddings.create(
 model=EMBEDDING_MODEL,
 input=texts,
 )
 return [item.embedding for item in response.data]



The `extract_facts` method sends conversation messages to the LLM with a carefully crafted prompt. The prompt tells the model to pull out only clear declarative statements and skip greetings, questions, and hypotheticals. Each extracted fact is a short, standalone sentence.

In [ ]:
 # -- Fact extraction --------------------------------------------------
 def extract_facts(self, messages: list[dict]) -> list[str]:
 """Use an LLM call to extract declarative facts from messages."""
 conversation_text = "\n".join(
 f"{m['role'].upper()}: {m['content']}" for m in messages
 )
 response = client.chat.completions.create(
 model=CHAT_MODEL,
 messages=[{
 "role": "user",
 "content": self.EXTRACTION_PROMPT.format(conversation=conversation_text),
 }],
 temperature=0.0,
 )
 raw = response.choices[0].message.content.strip()
 if raw == "NONE":
 return []
 # Each line is one fact
 facts = [line.strip().lstrip("- ") for line in raw.splitlines() if line.strip()]
 return facts

### Part 3: Deduplication and contradiction detection

Imagine two people file the same customer complaint using different words. A good clerk spots they're about the same issue. That's deduplication. Now imagine a customer says "I love the product" in January and "I hate the product" in March. The clerk needs to flag the contradiction and keep the recent one.

We handle both in `_find_match`. For each new fact, we compute its embedding and compare against every stored fact. High similarity (above 0.85) means it's a duplicate. Moderate similarity (above 0.50) means the facts are on the same topic, so we ask the LLM if they contradict each other.

In [ ]:
def _find_match(self, new_text: str, new_embedding: list[float]):
 """Check if a new fact duplicates or contradicts an existing one.

 Returns:
 ("duplicate", existing_fact) if the fact already exists.
 ("contradiction", existing_fact) if it conflicts with a stored fact.
 ("new", None) if it's a genuinely new fact.
 """
 best_sim = 0.0
 best_fact = None

 for fact in self.facts:
 if fact.archived:
 continue
 sim = cosine_similarity(new_embedding, fact.embedding)
 if sim > best_sim:
 best_sim = sim
 best_fact = fact

 if best_sim >= self.similarity_threshold:
 return ("duplicate", best_fact)

 if best_sim >= self.contradiction_threshold and best_fact is not None:
 # Same topic but different content. Ask the LLM if they contradict.
 if self._check_contradiction(best_fact.content, new_text):
 return ("contradiction", best_fact)

 return ("new", None)


def _check_contradiction(self, fact_a: str, fact_b: str) -> bool:
 """Ask the LLM whether two facts contradict each other."""
 response = client.chat.completions.create(
 model=CHAT_MODEL,
 messages=[{
 "role": "user",
 "content": self.CONTRADICTION_PROMPT.format(fact_a=fact_a, fact_b=fact_b),
 }],
 temperature=0.0,
 max_tokens=5,
 )
 answer = response.choices[0].message.content.strip().upper()
 return answer.startswith("YES")


# Attach methods to the class
SemanticMemory._find_match = _find_match
SemanticMemory._check_contradiction = _check_contradiction

### Part 4: Processing, retrieval, and chat

Now we wire everything together. `process_conversation` extracts facts from messages and routes each one through add, merge, or conflict resolution. `retrieve` finds the most relevant facts for a query. `chat` ties it all together: it retrieves relevant knowledge, injects it into the system prompt, and calls the LLM.

In [ ]:
def process_conversation(self, messages: list[dict]) -> dict:
 """Extract facts from messages and update the knowledge base.

 Returns a summary of what happened: counts of new, merged, and resolved facts.
 """
 extracted = self.extract_facts(messages)
 if not extracted:
 return {"extracted": 0, "new": 0, "merged": 0, "contradictions": 0}

 embeddings = self._embed_batch(extracted)
 stats = {"extracted": len(extracted), "new": 0, "merged": 0, "contradictions": 0}

 for fact_text, embedding in zip(extracted, embeddings):
 match_type, existing = self._find_match(fact_text, embedding)

 if match_type == "new":
 self.facts.append(Fact(content=fact_text, embedding=embedding))
 stats["new"] += 1

 elif match_type == "duplicate":
 # Boost confidence and update timestamp
 existing.mention_count += 1
 existing.confidence = min(1.0, existing.confidence + 0.05)
 existing.last_confirmed = datetime.now(timezone.utc).isoformat()
 stats["merged"] += 1

 elif match_type == "contradiction":
 # Archive the old fact, add the new one
 existing.archived = True
 self.facts.append(Fact(
 content=fact_text,
 embedding=embedding,
 confidence=0.9, # recent statements get high confidence
 ))
 stats["contradictions"] += 1

 return stats




The `retrieve` method finds stored facts relevant to a query using cosine similarity. The `chat` method ties everything together: it retrieves relevant facts, injects them into the system prompt, and calls the LLM. `get_active_facts` returns all non-archived facts for inspection.

In [ ]:
def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Fact, float]]:
 """Find the most relevant facts for a query using cosine similarity."""
 query_embedding = self._embed(query)
 scored = []
 for fact in self.facts:
 if fact.archived:
 continue
 sim = cosine_similarity(query_embedding, fact.embedding)
 scored.append((fact, sim))
 scored.sort(key=lambda x: x[1], reverse=True)
 return scored[:top_k]


def chat(self, user_input: str, conversation_history: list[dict] | None = None) -> str:
 """Respond to the user with relevant facts injected into the prompt."""
 relevant = self.retrieve(user_input, top_k=5)

 # Build system prompt with known facts
 system_parts = ["You are a helpful assistant."]
 if relevant:
 facts_text = "\n".join(
 f"- {fact.content} (confidence: {fact.confidence:.2f})"
 for fact, _sim in relevant
 )
 system_parts.append(
 f"\nHere is what you know about the user:\n{facts_text}\n"
 "Use these facts naturally. Don't list them unless asked."
 )

 messages = [{"role": "system", "content": "\n".join(system_parts)}]
 if conversation_history:
 messages.extend(conversation_history)
 messages.append({"role": "user", "content": user_input})

 response = client.chat.completions.create(
 model=CHAT_MODEL,
 messages=messages,
 )
 return response.choices[0].message.content


def get_active_facts(self) -> list[Fact]:
 """Return all non-archived facts."""
 return [f for f in self.facts if not f.archived]


# Attach methods to the class
SemanticMemory.process_conversation = process_conversation
SemanticMemory.retrieve = retrieve
SemanticMemory.chat = chat
SemanticMemory.get_active_facts = get_active_facts

## Example Run

Let's walk through a realistic scenario. A user chats with an agent across three sessions. Session one: introductions. Session two: work details. Session three: a preference change. We'll see the knowledge base grow, deduplicate, and resolve a contradiction.

### Session 1: Getting to know the user

The user shares basic information. The fact extractor pulls out declarative statements.

In [ ]:
memory = SemanticMemory()

session_1 = [
 {"role": "user", "content": "Hi! I'm Maya. I'm a data scientist living in Berlin."},
 {"role": "assistant", "content": "Nice to meet you, Maya! Berlin is a great city for tech. What are you working on?"},
 {"role": "user", "content": "I'm building a recommendation engine for an e-commerce company. We use Python and PyTorch mostly."},
 {"role": "assistant", "content": "Sounds like a solid stack. How large is your dataset?"},
 {"role": "user", "content": "About 50 million user interactions. We store everything in PostgreSQL and train on AWS."},
]

stats = memory.process_conversation(session_1)
print("Session 1 results:", stats)
print(f"\nKnowledge base now has {len(memory.get_active_facts())} facts:")
for fact in memory.get_active_facts():
 print(f" [{fact.confidence:.2f}] {fact.content}")

### Session 2: More details (with duplicates)

The user mentions some of the same information again. The system should merge these duplicates and boost their confidence instead of creating new entries.

In [ ]:
session_2 = [
 {"role": "user", "content": "I'm back! So we talked about my recommendation project before."},
 {"role": "assistant", "content": "Welcome back, Maya! Yes, the e-commerce recommendation engine. How's it going?"},
 {"role": "user", "content": "Good! My team has 5 engineers. We deploy our models on AWS SageMaker."},
 {"role": "assistant", "content": "SageMaker is a nice fit for that. Any challenges?"},
 {"role": "user", "content": "Latency is our biggest issue. We need sub-100ms inference for real-time recommendations."},
]

stats = memory.process_conversation(session_2)
print("Session 2 results:", stats)
print(f"\nKnowledge base now has {len(memory.get_active_facts())} active facts:")
for fact in memory.get_active_facts():
 confirmed = "(boosted)" if fact.mention_count > 1 else ""
 print(f" [{fact.confidence:.2f}] {fact.content} {confirmed}")

### Session 3: A contradiction

The user has moved cities. This contradicts a stored fact. The system should archive the old fact ("lives in Berlin") and store the new one ("lives in Amsterdam").

In [ ]:
session_3 = [
 {"role": "user", "content": "Hey, big news! I moved to Amsterdam last month for a new role."},
 {"role": "assistant", "content": "Congratulations on the move! What's the new role?"},
 {"role": "user", "content": "I'm now a senior ML engineer at a fintech startup. Still using Python though!"},
]

stats = memory.process_conversation(session_3)
print("Session 3 results:", stats)
print(f"\nActive facts ({len(memory.get_active_facts())}):")
for fact in memory.get_active_facts():
 print(f" [{fact.confidence:.2f}] {fact.content}")

archived = [f for f in memory.facts if f.archived]
if archived:
 print(f"\nArchived facts ({len(archived)}):")
 for fact in archived:
 print(f" [archived] {fact.content}")

### Retrieval: finding relevant facts

When the user asks a question, the system retrieves facts relevant to that query. Let's test with a few different queries.

In [ ]:
test_queries = [
 "What tech stack does Maya use?",
 "Where does the user live?",
 "Tell me about the user's team.",
]

for query in test_queries:
 print(f"Query: {query}")
 results = memory.retrieve(query, top_k=3)
 for fact, score in results:
 print(f" [{score:.3f}] {fact.content}")
 print()

### Chat with memory

Now let's see the full loop. The agent uses retrieved facts to answer questions it was never directly asked in this session.

In [ ]:
questions = [
 "Can you recommend a good deep learning conference near me?",
 "What programming language should I use for my next project?",
 "What do you know about me?",
]

for q in questions:
 print(f"User: {q}")
 reply = memory.chat(q)
 print(f"Agent: {reply}\n")

### Persistence

A semantic memory that disappears on restart isn't useful. We serialize the knowledge base to JSON so it survives between sessions.

In [ ]:
def save_memory(mem: SemanticMemory, path: str) -> None:
 """Save the knowledge base to a JSON file."""
 data = [asdict(f) for f in mem.facts]
 with open(path, "w") as f:
 json.dump(data, f, indent=2)
 active = len(mem.get_active_facts())
 print(f"Saved {active} active facts to {path}")


def load_memory(path: str, **kwargs) -> SemanticMemory:
 """Load a knowledge base from a JSON file."""
 with open(path) as f:
 data = json.load(f)
 mem = SemanticMemory(**kwargs)
 for item in data:
 mem.facts.append(Fact(**item))
 active = len(mem.get_active_facts())
 print(f"Loaded {active} active facts from {path}")
 return mem


# Round-trip test
save_memory(memory, "semantic_memory.json")
loaded = load_memory("semantic_memory.json")

# Verify the loaded memory works
reply = loaded.chat("Where do I live now?")
print(f"\nAgent (from loaded memory): {reply}")

Clean up the temporary file.

In [ ]:
if os.path.exists("semantic_memory.json"):
 os.remove("semantic_memory.json")

## Tradeoffs

### When Semantic Memory Works Well

- **Long-lived agents.** If your agent talks to the same user across many sessions, semantic memory accumulates a rich profile. The agent feels personalized without the user repeating themselves.
- **Cross-session knowledge.** Facts extracted in January are available in July. Buffer memory and sliding window memory lose this information when the session ends.
- **Efficient context use.** Instead of re-sending hundreds of past messages, you inject 5-10 relevant facts. This saves tokens and keeps the prompt focused.

### When It Breaks Down

- **Extraction errors.** The LLM may extract opinions as facts, miss implicit information, or hallucinate facts that weren't stated. Every fact in the knowledge base needs to have actually been said.
- **Subtle contradictions.** "I work remotely" and "I go to the office on Tuesdays" aren't strict contradictions, but the system may struggle with this nuance.
- **Scaling.** Comparing every new fact against every stored fact is O(n). For thousands of facts, you'll need approximate nearest neighbor search (ANN) instead of brute-force cosine similarity.
- **Stale facts.** Without a mechanism for temporal decay (reducing confidence over time), outdated facts linger. The user may have switched jobs six months ago, but the old job title persists.

### Cost Considerations

Each conversation turn costs one LLM call for extraction, one embedding call per extracted fact, and possibly one LLM call per contradiction check. For a turn that produces 3 facts, that's roughly 2-5 API calls. This is more expensive than buffer memory per turn, but cheaper per session when sessions are long.

## Further Reading

- [Tulving, "Episodic and Semantic Memory," *Organization of Memory*, 1972](https://psycnet.apa.org/record/1973-08477-005) - The original chapter where Endel Tulving defined semantic memory as a distinct system from episodic memory.
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," UIST 2023 (arXiv:2304.03442)](https://arxiv.org/abs/2304.03442) - Demonstrates both episodic and semantic memory in simulated agents that form beliefs from experiences.
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings) - Official documentation for the embeddings API used in this notebook.
- [Mem0: Long-Term Memory for AI Agents](https://github.com/mem0ai/mem0?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - An open-source library that implements fact extraction, deduplication, and contradiction detection.
- [Zep: Long-Term Memory for AI Assistants](https://github.com/getzep/zep?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - A memory server that extracts facts from conversations and provides semantic search over accumulated knowledge.

---

*Previous: [09 - Episodic Memory](../09_episodic_memory/) | Next: [11 - Procedural Memory](../11_procedural_memory/) ->*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Threshold sensitivity
Adjust the duplicate detection threshold (default 0.85) to 0.75, 0.80, 0.90, and 0.95. For each setting, process the same 20-turn conversation and count how many facts are classified as new, duplicate, or contradiction. Plot a confusion matrix for each threshold.

### Challenge 2: Fact accuracy audit
After processing 20 turns, export all active facts with `get_active_facts()`. Manually label each fact as correct, outdated, or wrong. Compute the accuracy rate. Identify whether errors come from extraction, deduplication, or contradiction resolution.

### Challenge 3: Categorized fact store
Group facts into named categories (e.g., 'personal', 'professional', 'preferences') by adding a `category` field to the `Fact` dataclass. Route facts to categories using the classification approach from 17 Memory Routing. Compare retrieval quality when searching within a category vs. searching all facts.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--10-semantic-memory--semantic-memory)
